# Credit Agreement Analyzer - Phase 3 Validation
Testing retrieval layer (embedder, vector store, BM25, hybrid) against a real credit agreement.

In [1]:
import time
from collections import Counter
from pathlib import Path

from credit_analyzer.processing.chunker import Chunker
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation"))

c:\Users\kbott\Projects\credit-analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0: Rebuild Phase 1 Outputs
Re-run extraction, section detection, definitions, and chunking.

In [2]:
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)
print(f"Pages: {doc.total_pages}")

detector = SectionDetector()
sections = detector.detect_sections(doc)
print(f"Sections: {len(sections)}")

defn_sections = [s for s in sections if s.section_type == "definitions"]
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0]) if defn_sections else None
assert defn_index is not None, "No definitions section found"
print(f"Defined terms: {len(defn_index.definitions)}")

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)
print(f"Chunks: {len(chunks)}")

Pages: 289
Sections: 156
Defined terms: 391
Chunks: 710


## Step 1: Embedder
Embed all chunks and verify dimensions, timing, and basic sanity.

In [3]:
embedder = Embedder()
print(f"Model dimension: {embedder.dimension}")

start = time.time()
embeddings = embedder.embed([c.text for c in chunks])
elapsed = time.time() - start

print(f"Embedded {len(embeddings)} chunks in {elapsed:.1f}s")
print(f"Embedding dim: {len(embeddings[0])}")
print(f"Rate: {len(embeddings) / elapsed:.0f} chunks/sec")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1463.56it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model dimension: 384
Embedded 710 chunks in 23.0s
Embedding dim: 384
Rate: 31 chunks/sec


In [4]:
# Sanity: query embedding should have same dimension
q_emb = embedder.embed_query("What is the restricted payments basket?")
print(f"Query embedding dim: {len(q_emb)}")
assert len(q_emb) == len(embeddings[0]), "Dimension mismatch!"

Query embedding dim: 384


## Step 2: Vector Store
Store chunks + embeddings in ChromaDB and test retrieval.

In [5]:
store = VectorStore(persist_directory=CHROMA_DIR)
# Delete any stale collection from previous runs
try:
    store.delete_collection("ribbon_test")
except Exception:
    pass
store.create_collection("ribbon_test")
store.add_chunks("ribbon_test", chunks, embeddings)
print(f"Stored {len(chunks)} chunks in ChromaDB")
print(f"Documents in store: {store.list_documents()}")

Stored 710 chunks in ChromaDB
Documents in store: ['ribbon_test']


In [6]:
# Test: semantic search for restricted payments
q = "What is the restricted payments basket?"
q_emb = embedder.embed_query(q)
results = store.search("ribbon_test", q_emb, top_k=5)

print(f"Query: {q}")
print(f"Results: {len(results)}\n")
for r in results:
    print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title} ({r.chunk.section_type})")
    print(f"  Text: {r.chunk.text[:150]}...\n")

Query: What is the restricted payments basket?
Results: 5

  Score: 0.882 | 1.1 - Defined Terms (definitions)
  Text: “Restricted Payments”: is defined in Section 7.6 and do not include, for the avoidance of doubt, amounts paid in-kind with
respect to preferred Capita...

  Score: 0.865 | 1.1 - Defined Terms (definitions)
  Text: “General Letter of Credit Basket”: as defined in Section 7.2(r)....

  Score: 0.861 | 7.6 - Restricted Payments (negative_covenants)
  Text: 7.6
Restricted Payments. Make any payment with respect to any Deferred Payment Obligation, make any payment or
prepayment of principal of, premium, if...

  Score: 0.850 | 1.1 - Defined Terms (definitions)
  Text: “Starter Basket”: is defined in clause (a) of the definition of “Available Amount”.

“Subordinated Debt Document”: any agreement, certificate, documen...

  Score: 0.848 | 7.6 - Restricted Payments (negative_covenants)
  Text: (a)

(i) any Subsidiary may make Restricted Payments to any Loan Party, and (ii) any S

In [7]:
# Test: filtered search within negative covenants only
q = "How much incremental debt can the borrower incur?"
q_emb = embedder.embed_query(q)
results = store.search("ribbon_test", q_emb, top_k=5, section_filter="negative_covenants")

print(f"Query: {q}")
print("Filter: negative_covenants")
print(f"Results: {len(results)}\n")
for r in results:
    print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title}")
    print(f"  Text: {r.chunk.text[:150]}...\n")

Query: How much incremental debt can the borrower incur?
Filter: negative_covenants
Results: 5

  Score: 0.852 | 7.2 - Indebtedness
  Text: 7.2
Indebtedness. Create, issue, incur, assume, become liable in respect of or suffer to exist any Indebtedness, except:

(a)
Indebtedness of any Loan...

  Score: 0.847 | 7.2 - Indebtedness
  Text: (g)

(i) Permitted Ratio Debt and (ii) Credit Agreement Refinancing Indebtedness;

(h)
Indebtedness of Holdings and its Subsidiaries in an aggregate p...

  Score: 0.846 | 7.8 - Investments
  Text: (o)
[reserved];

(p)
intercompany cost-plus or transfer pricing transactions in connection with the ongoing business operations of
Subsidiaries of Hol...

  Score: 0.844 | 7.2 - Indebtedness
  Text: For purposes of determining compliance with this Section 7.2, in the event that an item of Indebtedness (or any portion
thereof) meets the criteria of...

  Score: 0.836 | 7.2 - Indebtedness
  Text: (r)
Indebtedness incurred by the Borrower or any of its Subsidiar

## Step 3: BM25 Store
Test keyword search for exact terms, dollar amounts, and ratios.

In [8]:
bm25 = BM25Store()
bm25.build_index(chunks)
print(f"BM25 index built over {len(chunks)} chunks")

BM25 index built over 710 chunks


In [9]:
# Test: exact dollar amount search
for q in ["$50,000,000", "4.75:1.00", "Incremental Facility", "Restricted Payments"]:
    results = bm25.search(q, top_k=3)
    print(f"Query: {q}")
    if results:
        for r in results:
            print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title} ({r.chunk.chunk_type})")
    else:
        print("  No results")
    print()

Query: $50,000,000
  Score: 12.135 | 1.1 - Defined Terms (definition)
  Score: 11.771 | 1.1 - Defined Terms (definition)
  Score: 11.462 | 1.1 - Defined Terms (definition)

Query: 4.75:1.00
  Score: 17.630 | 7.1 - Financial Condition Covenants (table)
  Score: 17.406 | 7.1 - Financial Condition Covenants (text)
  Score: 9.431 | 10.23 - Acknowledgment Regarding any Supported QFCs (text)

Query: Incremental Facility
  Score: 21.818 | 2.27 - Incremental Facility (text)
  Score: 21.801 | 2.27 - Incremental Facility (text)
  Score: 20.512 | 2.27 - Incremental Facility (text)

Query: Restricted Payments
  Score: 18.724 | 7.6 - Restricted Payments (text)
  Score: 17.266 | 7.6 - Restricted Payments (text)
  Score: 16.815 | 7.6 - Restricted Payments (text)



In [10]:
# Test: BM25 with section filter
results = bm25.search("Restricted Payments", top_k=5, section_filter="negative_covenants")
print(f"BM25 'Restricted Payments' filtered to negative_covenants: {len(results)} results")
for r in results:
    print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title}")

BM25 'Restricted Payments' filtered to negative_covenants: 5 results
  Score: 9.057 | 7.6 - Restricted Payments
  Score: 8.487 | 7.6 - Restricted Payments
  Score: 8.216 | 7.6 - Restricted Payments
  Score: 8.123 | 7.6 - Restricted Payments
  Score: 5.633 | 7.16 - Clauses Restricting Subsidiary Distributions


## Step 4: Hybrid Retriever
Test combined vector + BM25 retrieval with definition injection.

In [11]:
retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)

In [12]:
# Test query 1: Incremental debt capacity (your core use case)
result = retriever.retrieve(
    query="How much incremental debt can the borrower incur?",
    document_id="ribbon_test",
    top_k=5,
)

print("Query: How much incremental debt can the borrower incur?")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

print(f"Injected definitions ({len(result.injected_definitions)}):")
for term, defn in result.injected_definitions.items():
    print(f"  {term}: {defn[:100]}...")

Query: How much incremental debt can the borrower incur?
Chunks returned: 5

  Score: 0.973 | Source: both   | 2.27 - Incremental Facility
  Text: 2.27
Incremental Facility.

(a)
At any time after the Closing Date during the Commitment Period (in the case of a Revolving Commitment Increase) or any
time after the Closing Date and prior to the Ter...

  Score: 0.600 | Source: vector | 1.1 - Defined Terms
  Text: “Ratio Incremental Amount”: an aggregate principal amount of Indebtedness that, immediately after the incurrence thereof on Pro Forma
Basis, would not result in, (a) with respect to Incremental Facili...

  Score: 0.569 | Source: both   | 2.27 - Incremental Facility
  Text: (b)
Each of the following shall be conditions precedent to the effectiveness of any Incremental Facility:

(i)
the Borrower shall have delivered an irrevocable written request to the Administrative Ag...

  Score: 0.264 | Source: both   | 1.1 - Defined Terms
  Text: “Term Commitment”: as to any Lender, the obl

In [13]:
# Test query 2: Financial covenants
result = retriever.retrieve(
    query="What are the financial covenant test levels?",
    document_id="ribbon_test",
    top_k=5,
)

print("Query: What are the financial covenant test levels?")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Type: {hc.chunk.chunk_type}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

print(f"Injected definitions ({len(result.injected_definitions)}):")
for term in result.injected_definitions:
    print(f"  {term}")

Query: What are the financial covenant test levels?
Chunks returned: 5

  Score: 0.600 | Source: vector | 7.1 - Financial Condition Covenants
  Type: text
  Text: 7.1
Financial Condition Covenants.

(a)
Maximum Consolidated Net Leverage Ratio. Commencing September 30, 2024, permit the Consolidated Net
Leverage Ratio, as at the last day of any period of four con...

  Score: 0.431 | Source: both   | 7.1 - Financial Condition Covenants
  Type: text
  Text: (b)
Notwithstanding anything to the contrary in Section 7.1, in the event the Borrower fails to comply with Section 7.1(a) as
of the last day of any period of four consecutive trailing fiscal quarters...

  Score: 0.429 | Source: vector | 10.23 - Acknowledgment Regarding any Supported QFCs
  Type: text
  Text: 30.
The following is a true and correct list of all letters of credit as to which any Loan Party is the beneficiary or otherwise has
any right to payment or performance: None.
INFORMATION ABOUT THE LO...

  Score: 0.414 | Source:

In [14]:
# Test query 3: Specific dollar amount (BM25 should shine here)
result = retriever.retrieve(
    query="What is the revolving commitment amount?",
    document_id="ribbon_test",
    top_k=5,
)

print("Query: What is the revolving commitment amount?")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

Query: What is the revolving commitment amount?
Chunks returned: 5

  Score: 1.000 | Source: both   | 1.1 - Defined Terms
  Text: “Total Revolving Commitments”: at any time, the aggregate amount of the Revolving Commitments then in effect. The
original amount of the Total Revolving Commitments is $35,000,000. The L/C Commitment ...

  Score: 0.541 | Source: both   | 1.1 - Defined Terms
  Text: “Revolving Commitment”: as to any Lender, the obligation of such Lender, if any, to make Revolving Loans in an aggregate
principal amount not to exceed the amount set forth under the heading “Revolvin...

  Score: 0.362 | Source: both   | 2.4 - Revolving Commitments
  Text: 2.4
Revolving Commitments.
Subject to the terms and conditions hereof, each Revolving Lender severally agrees to make revolving credit loans (each, a
“Revolving Loan” and, collectively, the “Revolving...

  Score: 0.316 | Source: vector | 1.1 - Defined Terms
  Text: “Revolving Commitment Increase”: is defined in Section 2.27(a

In [15]:
# Test query 4: With section filter (report generation pattern)
result = retriever.retrieve(
    query="What restricted payments are permitted?",
    document_id="ribbon_test",
    top_k=5,
    section_filter="negative_covenants",
)

print("Query: What restricted payments are permitted? (filtered: negative_covenants)")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

print(f"Injected definitions ({len(result.injected_definitions)}):")
for term in result.injected_definitions:
    print(f"  {term}")

Query: What restricted payments are permitted? (filtered: negative_covenants)
Chunks returned: 5

  Score: 0.800 | Source: both   | 7.6 - Restricted Payments
  Text: 7.6
Restricted Payments. Make any payment with respect to any Deferred Payment Obligation, make any payment or
prepayment of principal of, premium, if any, or interest on, or redemption, purchase, ret...

  Score: 0.738 | Source: both   | 7.6 - Restricted Payments
  Text: (a)

(i) any Subsidiary may make Restricted Payments to any Loan Party, and (ii) any Subsidiary that is not a Loan
Party may make Restricted Payments to any other Group Member or, to any other holder ...

  Score: 0.340 | Source: both   | 7.16 - Clauses Restricting Subsidiary Distributions
  Text: 7.16
Clauses Restricting Subsidiary Distributions. Enter into or suffer to exist or become effective any consensual
encumbrance or restriction on the ability of any Subsidiary of Holdings to (a) make ...

  Score: 0.281 | Source: both   | 7.6 - Restricted Paymen

In [16]:
# Check source distribution across all test queries
test_queries = [
    "How much incremental debt can the borrower incur?",
    "What are the financial covenant test levels?",
    "What is the revolving commitment amount?",
    "What restricted payments are permitted?",
    "Who are the administrative agents?",
    "What are the mandatory prepayment triggers?",
]

source_counts: Counter[str] = Counter()
for q in test_queries:
    r = retriever.retrieve(q, "ribbon_test", top_k=5)
    for hc in r.chunks:
        source_counts[hc.source] += 1

print("Source distribution across test queries:")
total = sum(source_counts.values())
for source, count in source_counts.most_common():
    print(f"  {source}: {count} ({count/total*100:.0f}%)")

Source distribution across test queries:
  both: 15 (50%)
  vector: 10 (33%)
  bm25: 5 (17%)


## Cleanup

In [17]:
store.delete_collection("ribbon_test")
print("Cleaned up test collection")

Cleaned up test collection
